# How to conduct experiments

Every time you change a layer, a learning rate or a batch size, you are running an experiment. This notebook shows how to run experiments you (and others) can trust: repeatable, reproducible and honestly reported.

Use it as the checklist for your course project. The worked example at the end is the minimum workflow expected there.

### Background

Empirical research in machine learning rests on a simple premise. A reported result should reflect a real property of the method under study. An experiment that cannot be repeated cannot be verified. Reproducibility is therefore a precondition of valid knowledge in the discipline.

Repeatability means the same team obtains the same results using the same experimental setup. 

Reproducibility means an independent team obtains the same results using the original artefacts, such as code, data and configuration. 

Replicability means an independent team reaches the same conclusions using its own, independently developed artefacts.

Deep learning is especially vulnerable to failures of reproducibility, for several reasons.

1. Stochasticity is pervasive in the training process. It enters through parameter initialisation, data shuffling and batching, dropout, data augmentation, and the partitioning of data into training, validation and test sets.

2. Computation on graphics processing units (GPUs) is not necessarily deterministic. Parallel floating-point reductions may be executed in varying order, so numerically different results can arise from nominally identical runs.

3. Results depend strongly on the software and hardware environment, including library versions, low-level numerical libraries (such as CUDA and cuDNN) and the processor architecture.

4. Deep learning models have a large number of hyperparameters. These are frequently tuned informally and documented incompletely.

### Principles

1. Control of Randomness

The first requirement is to fix every source of pseudo-randomness. In Keras 3, the function keras.utils.set_random_seed initialises the random number generators of Python, NumPy and the active computational backend in a single call. Where the TensorFlow backend is used, deterministic execution of GPU operations must also be enabled explicitly. This is done at some cost to computational speed.

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "0"

import keras
import tensorflow as tf

SEED = 42
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

A global seed does not exempt the researcher from seeding individual stochastic components explicitly. Doing so makes the intended behaviour visible in the code itself:

In [ ]:
keras.layers.Dense(64, kernel_initializer=keras.initializers.GlorotUniform(seed=SEED))
keras.layers.Dropout(0.3, seed=SEED)
keras.layers.RandomFlip("horizontal", seed=SEED)

Data shuffling must likewise be seeded. Examples are dataset.shuffle(buffer_size, seed=SEED) in tf.data pipelines and random_state=SEED in scikit-learn utilities.

Remember to mention your seeds in the report, presentation or paper. The reproducibility needs to know, what seeds you used in the process.

2. Data Management

The partition of data into training, validation and test sets should be created once, recorded and reused across all experiments. Regenerating it at each run introduces an uncontrolled source of variation. All data-dependent preprocessing must be fitted exclusively on the training set. This covers normalisation statistics, vocabularies and imputation values. For example, a keras.layers.Normalization layer should be adapted only to training data. The researcher should also check for the less obvious forms of leakage: the same entity appearing in several partitions (for example, multiple records of one patient), temporal leakage in time-dependent data, and duplicated observations.

The test set should be used only once, for final evaluation. All model selection and hyperparameter tuning must rely solely on the validation set. Finally, the version of the dataset used should be recorded, whether by a cryptographic hash, a data-versioning tool such as DVC, or at minimum a precise description of its source and date of acquisition.

3. Externalised configuration

All hyperparameters and experimental settings should be specified in a dedicated configuration file, for example in YAML or JSON format, rather than embedded throughout the source code. If embeded through the source code, it should be placed in one block, separate from the code itself.

Example in YAML:

```yaml
seed: 42
model:
  units: 128
  dropout: 0.3
train:
  learning_rate: 0.001
  batch_size: 32
  epochs: 50
  optimizer: adam
data:
  version: v1.2
  validation_split: 0.15
```

A copy of this file should be archived together with the results of each run. Every reported result then has a complete and unambiguous description of the conditions under which it was obtained.

4. Specification of the Computational Environment

The software environment must be recorded with enough precision to be reconstructed. At a minimum, the researcher should provide a list of all dependencies with exact versions (for example via pip freeze > requirements.txt or a conda environment.yml file). They should also record the versions of Python, Keras and the computational backend, together with the type of hardware used. For more demanding projects, containerisation (for example with Docker) offers the most complete guarantee of environmental reproducibility. When cloud notebooks such as Google Colab are used, the runtime type should be reported and package versions pinned explicitly at installation.

5. Systematic Logging and Experiment Tracking

The course of training and its outcomes should be recorded automatically and in full. Keras provides callback mechanisms that serve this purpose:

Example (uses `model`, `train_ds`, `val_ds` and `config` from your own project):

```python
callbacks = [
    keras.callbacks.ModelCheckpoint("runs/exp01/best.keras",
                                    monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                  restore_best_weights=True),
    keras.callbacks.CSVLogger("runs/exp01/history.csv"),
    keras.callbacks.TensorBoard(log_dir="runs/exp01/tensorboard"),
]

model.fit(train_ds, validation_data=val_ds,
          epochs=config["train"]["epochs"], callbacks=callbacks)
model.save("runs/exp01/final.keras")
```

6. Evaluation and Reporting

Reproducibility of execution does not by itself ensure validity of conclusions. A single training run is a single sample from a distribution of possible outcomes. Each configuration should therefore be evaluated over several independent random seeds; three to five is a practical minimum.

The extent of the hyperparameter search, that is, the number of configurations evaluated, should be disclosed. Where the differences between methods are small, they should be assessed with confidence intervals or statistical tests. A complete report also gives the computational cost of the experiments, the hardware used, and any configurations that failed or were abandoned.

## Worked Example: A Reproducible Experiment in Seven Steps

This example shows the minimum workflow expected in course projects. The task is image classification on the Fashion-MNIST dataset with a small neural network in Keras. All code can go in a single notebook or script and should be run from top to bottom.

### Step 1: Record the environment

Before starting, save the exact versions of the libraries you use:

In [ ]:
!pip freeze > requirements.txt

Submit this file with your project.

### Step 2: Collect all settings in one place

Put every hyperparameter at the top of the code, so that nothing is hidden elsewhere:

In [ ]:
CONFIG = {
    "split_seed": 2026,      # fixed: the data split never changes
    "seeds": [123, 2137, 15190],      # training is repeated with these seeds
    "val_fraction": 0.1,
    "dense_units": 128,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 10,
}

### Step 3: Fix the sources of randomness

This function fixes all random number generators and makes GPU computations deterministic:

In [ ]:
import keras
import numpy as np
import tensorflow as tf

def set_seed(seed):
    keras.utils.set_random_seed(seed)              # Python, NumPy, TensorFlow
    tf.config.experimental.enable_op_determinism()  # deterministic GPU ops

### Step 4: Prepare the data correctly

Two rules apply here:
1. The training/validation split is made once, with a fixed seed.
2. Normalisation statistics are computed **on the training data only**.

In [ ]:
(x_full, y_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Fixed split: the same every time the code is run
rng = np.random.default_rng(CONFIG["split_seed"])
idx = rng.permutation(len(x_full))
n_val = int(len(idx) * CONFIG["val_fraction"])
val_idx, train_idx = idx[:n_val], idx[n_val:]

x_train, y_train = x_full[train_idx], y_full[train_idx]
x_val, y_val = x_full[val_idx], y_full[val_idx]

# Normalise using training statistics only (prevents data leakage)
mean, std = x_train.mean(), x_train.std()
x_train = ((x_train - mean) / std).astype("float32")
x_val = ((x_val - mean) / std).astype("float32")
x_test = ((x_test - mean) / std).astype("float32")

### Step 5: Define a single training run

One function builds, trains and evaluates the model for a given seed. The training history is saved to a file. The test set is used only once, at the very end.

In [ ]:
def run_experiment(seed):
    set_seed(seed)

    model = keras.Sequential([
        keras.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(CONFIG["dense_units"], activation="relu"),
        keras.layers.Dropout(CONFIG["dropout"], seed=seed),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(CONFIG["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        callbacks=[keras.callbacks.CSVLogger(f"history_seed_{seed}.csv")],
        verbose=0,
    )
    model.save(f"model_seed_{seed}.keras")

    _, test_acc = model.evaluate(x_test, y_test, verbose=0)
    return test_acc

### Step 6: Repeat over several seeds and report mean ± standard deviation

One run is not enough to draw a conclusion. Repeat the experiment and report the average and spread:

In [ ]:
results = [run_experiment(s) for s in CONFIG["seeds"]]

for s, acc in zip(CONFIG["seeds"], results):
    print(f"Seed {s}: test accuracy = {acc:.4f}")

print(f"Mean ± std: {np.mean(results):.4f} ± {np.std(results, ddof=1):.4f}")

Report this mean ± std value in your project, not the best single run.

### Step 7: Check that it is reproducible

Run the same seed twice. The results should be identical:

In [ ]:
a = run_experiment(1)
b = run_experiment(1)
print("Reproducible:", a == b)

If the result is `False`, some source of randomness is not controlled. Find it before reporting any results.

### What to submit

- The notebook or script, runnable from top to bottom.
- `requirements.txt`.
- The training histories (`history_seed_*.csv`).
- A short results table (mean ± std over the seeds).